# F1 · Forgetting Curve — CoxPH Survival Analysis
Predicting flashcard recall probability using survival analysis on review history.

In [ ]:
import django, os, sys
sys.path.insert(0, os.path.abspath('..'))
os.environ.setdefault('DJANGO_SETTINGS_MODULE', 'lumen_project.settings')
django.setup()

## 1. Dataset Description

The review feature dataset is derived from `FlashcardReview` records stored in the Django database.
Each row represents one flashcard review event with the following engineered features:

| Column | Description |
|---|---|
| `user_id` | Foreign key to the reviewing user |
| `topic` | Subject topic of the flashcard |
| `avg_grade` | Rolling average grade across all prior reviews (0–5) |
| `num_prior_reviews` | Count of previous reviews for this card |
| `hours_since_last_review` | Time elapsed since most recent review (hours) |
| `skill` | Estimated skill level derived from avg_grade |
| `topic_difficulty` | Aggregate difficulty score for the topic |
| `recalled` | Binary target: 1 = recalled correctly, 0 = forgotten |

In [ ]:
from analytics.ml.feature_engineering import get_review_features
df = get_review_features()
print(df.shape)
print(df.describe())

## 2. EDA

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from lifelines import KaplanMeierFitter

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Grade distribution bar chart
grade_counts = df['avg_grade'].round().value_counts().sort_index()
axes[0].bar(grade_counts.index, grade_counts.values, color='steelblue', edgecolor='white')
axes[0].set_title('Grade Distribution')
axes[0].set_xlabel('avg_grade (rounded)')
axes[0].set_ylabel('Count')

# Days since last review histogram
days = df['hours_since_last_review'] / 24
axes[1].hist(days, bins=30, color='coral', edgecolor='white')
axes[1].set_title('Days Since Last Review')
axes[1].set_xlabel('Days')
axes[1].set_ylabel('Count')

# Kaplan-Meier estimator for recall survival
kmf = KaplanMeierFitter()
kmf.fit(durations=days, event_observed=df['recalled'])
kmf.plot_survival_function(ax=axes[2], ci_show=True)
axes[2].set_title('KM Recall Survival Curve')
axes[2].set_xlabel('Days since last review')
axes[2].set_ylabel('P(recall)')

plt.tight_layout()
plt.savefig('../models/01_eda.png', dpi=100)
plt.show()

## 3. Baseline Model

The baseline uses a simple exponential decay formula (Ebbinghaus forgetting curve):

$$P(\text{recall}) = e^{-t / S}$$

where `t` is hours since last review and `S` is the stability constant estimated from the mean retention time in the dataset. Concordance index is used as the primary evaluation metric.

In [ ]:
import numpy as np
from lifelines.utils import concordance_index

# Estimate stability S as mean hours-to-forget across recalled=0 events
forgotten = df[df['recalled'] == 0]
S = forgotten['hours_since_last_review'].mean() if len(forgotten) > 0 else df['hours_since_last_review'].mean()
S = max(S, 1.0)  # guard against zero

# Baseline predicted recall probability
df['baseline_p'] = np.exp(-df['hours_since_last_review'] / S)

# Concordance index: higher score → better at ranking who will/won't recall
baseline_ci = concordance_index(
    df['hours_since_last_review'],
    -df['baseline_p'],          # negate because higher p_recall = shorter expected time
    df['recalled']
)
print(f'Baseline stability constant S = {S:.1f} hours')
print(f'Baseline concordance index    = {baseline_ci:.4f}')

## 4. CoxPH Model

Cox Proportional Hazards regression models the hazard of forgetting as a function of learner and topic covariates. This gives a personalised survival curve per review event.

In [ ]:
from lifelines import CoxPHFitter
import pandas as pd

cox_features = [
    'avg_grade', 'num_prior_reviews',
    'hours_since_last_review', 'skill', 'topic_difficulty'
]

cox_df = df[cox_features + ['recalled']].dropna()

# lifelines CoxPH expects duration + event columns
cox_df = cox_df.rename(columns={
    'hours_since_last_review': 'duration',
    'recalled': 'event'
})

cph = CoxPHFitter(penalizer=0.1)
cph.fit(cox_df, duration_col='duration', event_col='event')
cph.print_summary()

# Concordance index on training data
cox_ci = cph.concordance_index_
print(f'\nCoxPH concordance index = {cox_ci:.4f}')

## 5. Results Comparison

In [ ]:
import pandas as pd

results = pd.DataFrame([
    {'Model': 'Baseline (Exponential Decay)', 'Concordance Index': round(baseline_ci, 4),
     'Notes': 'Mean stability S; no covariates'},
    {'Model': 'CoxPH Survival Model',         'Concordance Index': round(cox_ci, 4),
     'Notes': 'avg_grade, num_prior_reviews, hours, skill, topic_difficulty'},
])

results = results.set_index('Model')
print(results.to_string())
print(f'\nImprovement: +{(cox_ci - baseline_ci):.4f} concordance index points')

## 6. Limitations

- Small dataset (synthetic): model coefficients have high variance; not suitable for real deployment without a larger review history corpus.
- Skill approximated from grade: true skill level requires IRT-style estimation across multiple topics.
- No temporal validation split: concordance index is computed in-sample, likely optimistic. A held-out time window (e.g., last 30 days) should be used for honest evaluation.
- CoxPH proportional-hazards assumption may not hold across very different topics (e.g., vocabulary vs. calculus).
- Censoring mechanism not verified: reviews that were never attempted are not included, introducing selection bias.

## 7. Production Usage

Use `predict_recall(user_id, topic)` from `analytics.ml.forgetting_curve`.

The model artifact is serialised to `models/forgetting_curve.pkl` by:

```
python manage.py train_forgetting_curve
```

See also: `analytics/management/commands/train_forgetting_curve.py`

The wrapper loads the fitted `CoxPHFitter`, constructs the covariate row for the given `(user_id, topic)` pair from live database data, and returns a dict with `p_recall` and `optimal_review_in_days`.

In [ ]:
from analytics.ml.forgetting_curve import predict_recall

# result = predict_recall(user_id=1, topic='calculus')
# print(result)  # {'p_recall': 0.823, 'optimal_review_in_days': 3}

# Dry-run: show function signature
import inspect
print(inspect.signature(predict_recall))